# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset package with the `mlcroissant` library using Croissant schema references throughout.

### Dataset Source
The dataset is described by a Croissant schema published at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load the Croissant metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs using the Croissant schema references.

**Note:** All record sets, fields, and columns are always referenced by their `@id` fields.

In [ ]:
# List all record sets by their @id and display their fields
print("Available record sets and their fields by @id:")
for record_set in dataset.record_sets:
    print(f"\nRecord Set @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
  
    print("  Fields:")
    for field in record_set.fields:
        col_names = [col.id for col in getattr(field, 'columns', [])]
        print(f"    Field @id: {field.id}")
        print(f"      Name: {field.name}")
        print(f"      dataType: {field.data_type if hasattr(field, 'data_type') else None}")
        if col_names:
            print(f"      Columns (by @id): {col_names}")
        else:
            print(f"      Columns: None")

## 3. Data Extraction

Now let's extract data for each available record set using its `@id`, placing each set into a separate pandas DataFrame.

For reference, see the above cell for record set and field `@id` values.

In [ ]:
# Generate a list of record set @ids from the dataset
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = dict()

for record_set_id in record_set_ids:
    # Use mlcroissant's records interface by @id
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records for {record_set_id}")
    else:
        print(f"No records found for {record_set_id}")

### Example: Examine the variables and first few rows for one record set

We select the (only/main) record set in the dataset. Replace the variable below by the desired `@id` if there are multiple record sets.

In [ ]:
# Choose the main record set @id from the loaded ones
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Main record set: {main_record_set_id}")
    main_df = dataframes[main_record_set_id]
    print("\nColumns available (field and column @ids):")
    print(main_df.columns.tolist())
    display(main_df.head())
else:
    print("No dataframes loaded!")

## 4. Exploratory Data Analysis (EDA)

Apply some data processing steps, such as filtering, normalization, and grouping. All operations refer to columns by their `@id`.

Below, select a numeric field for analysis (by its Croissant field/column `@id`).

In [ ]:
# Inspect available columns to find a numeric @id
print("Available columns (@id):", main_df.columns.tolist())

# Example: Suppose there is a column with @id 'age' (check actual @id from your dataset above).
# Replace this with the correct numeric field @id for your data, e.g. 'age' or similar.

numeric_field_id = None
for col in main_df.columns:
    # Heuristic: pick the first field containing 'age' or 'interval' or 'years' as numeric
    if 'age' in col.lower() or 'interval' in col.lower() or 'years' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Otherwise pick the first column (fallback)
    numeric_field_id = main_df.columns[0]

print(f"Selected numeric field (@id): {numeric_field_id}")

# Make sure the column is numeric
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

# For demonstration, filter for values above a threshold (e.g., 50 for age, or 1 for intervals)
threshold = main_df[numeric_field_id].quantile(0.75) if main_df[numeric_field_id].notnull().sum() > 0 else 0
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (75th percentile):")
display(filtered_df.head())

# Normalize the numeric field within the filtered DataFrame
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field (@id) if available
group_field_id = None
for col in main_df.columns:
    # Try common group indicators
    if any(s in col.lower() for s in ['sex', 'gender', 'site', 'location', 'status', 'type', 'category']):
        group_field_id = col
        break

if group_field_id:
    print(f"Grouping by {group_field_id} (by @id):")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    display(grouped_df)
else:
    print("No clear grouping field found by @id.")

## 5. Visualization

Visualize the distribution of the numeric field and (if possible) compare groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (filtered)
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, bins=10, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group if available
if group_field_id:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

- This notebook demonstrated accessing the FAIR² dataset via its Croissant schema and exploring its structure using @id references throughout.
- We loaded data, inspected available entities, filtered and normalized a numeric field, and visualized key features.
- All operations referenced entities by their Croissant `@id` fields to ensure clear reporting and reproducibility.

Continue your exploration and analysis by referencing the schema's `@id` fields for all dataset elements!